# Benchmarking Rust vs Python Mofography Periodic Segmentation Workflow
## Key points
- We need to make the playing field even. Essentially, allocating the same resources for both Rust and Python functions.
    - One cpu (core) and same number of threads per job.
    - One job is everything needed for one material. One dgrid calculation, and one do_periodic_segmentation call.
- Functions for Python: dgrid_from_atoms_cpu, do_periodic_segmentation
- Functions for Rust: dgrid_from_atoms_rust, do_periodic_segmentation_rust

**IMPORTANT NOTE**
- The Python functions are no slouch. If we let it, the dgrid calculation will run over all the cores also the connections calculation in python uses the ThreadPoolExecutor, so that could also run over ncores * nthreads in a node if we allow it. So, we need to wrap the dgrid in a wrapper function to stop it from using all the cores, and then wrap the wrapper in dask to make sure connections computing wont' use all the threads on all the cores, ensuring both run over the same resources, one core and N threads per core per material.
- So please make sure that all cpus are on the same node to avoid data transfer bottlenecks, and ncores = n_workers in the dask cluster when you run the workflow, then we can say we used 1 core per material. 


## There are a few more useful things in thie notebook, 
- It is painful to sample a set of ASE atoms from large ZIP full of 1000s of cif files. Especially when CIFs are big. So we'll put them in a parquet file (compressed), pick cifs first, then read in parallel spamming many dask workers, since each operation takes only little memory. See the details below.
- A function to plot whatever index material from the results list
- A function to plot dgrid_time and seg_time vs num_atoms, for rust vs python. 

## Grab a representatative subset of ASE atoms
When working with thousands of CIF files stored in a ZIP archive, extracting and parsing each file individually is extremely slow and I/O-bound. Each random access requires decompressing and reading from the archive, which does not scale well for large datasets. By converting all CIFs into a compressed Parquet DataFrame containing the raw CIF strings, we enable fast, parallel, and random access to any subset of structures. This approach dramatically reduces overhead and makes it practical to sample and process hundreds or thousands of structures efficiently, as demonstrated in the next cells.

### Convert zip file into easily searchable Parquet
The next cell demonstrates how to efficiently convert all CIF files from the `Tobacco.zip` archive into a compressed Parquet DataFrame containing the CIF strings. This approach allows for much faster loading and random access to CIF data in future analyses, compared to repeatedly extracting and parsing files from the ZIP archive. The cell uses parallel extraction with `ThreadPoolExecutor` and saves the resulting DataFrame with Zstd compression for optimal storage efficiency.

In [ ]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import zipfile

# Step 1: Extract CIF strings from ZIP in parallel
with zipfile.ZipFile('Tobacco.zip') as zf:
    cif_files = [f for f in zf.namelist() if f.lower().endswith('.cif')]
    
    def extract_cif_string(cif_file):
        with zf.open(cif_file) as f:
            cif_bytes = f.read()
            return cif_bytes.decode('latin-1')  # Use latin-1 to handle any encoding
    
    with ThreadPoolExecutor(max_workers=4) as executor:  # Adjust workers based on your CPU
        cif_strings = list(executor.map(extract_cif_string, cif_files))

# Step 2: Store in DataFrame
data = {
    'name': [f.replace('.cif', '') for f in cif_files],  # Remove .cif extension
    'cif': cif_strings
}
df = pd.DataFrame(data)

# Step 3: Save to compressed Parquet file with Zstd for max compression
df.to_parquet('tobacco_cifs.parquet', compression='zstd', index=False)
print(f"Converted {len(df)} CIF files to compressed Parquet dataframe with strings.")

### Random sample from Parquet file
This is an annoying step, because these CIF files are huge, its better to sample first and then read and convert them into ase atoms objects. This is a low memory operation, so we'll throw a lot of dask workers at them and read them as ase atoms objects rather quickly. 

#### Make imports

In [ ]:
# Alternative fast loading: Use the pre-converted Parquet dataframe with CIF strings
# Run this instead of the slow ZIP reading cells above
import pandas as pd
from dask import delayed
import dask
from dask.distributed import progress, Client
import warnings
from ase.io import read
from io import  BytesIO


### Set N 
How many to sample form the parquet

In [ ]:
# Sample N random rows
N = 10  # Adjust as needed

#### Start local cluster
Put a large number of workers here to speed up the file reading. Each of it takes only a small amount of RAM so we can spam workers to get the files read into ASE atoms quickly.

In [ ]:

# client.shutdown()
client = Client(n_workers=25)


In [ ]:
client

### Read and convert to ase atoms objects

In [ ]:
# Suppress warnings
warnings.filterwarnings('ignore')

# Load the compressed dataframe
df = pd.read_parquet('tobacco_cifs.parquet')


random_rows = df.sample(N, random_state=42)  # Use random_state for reproducibility

# Function to parse CIF string to atoms
def cif_string_to_atoms(cif_string):
    cif_bytes = cif_string.encode('latin-1')  # Use latin-1 for consistency
    return read(BytesIO(cif_bytes), format='cif')

# Parse selected CIF strings to atoms in parallel using Dask delayed (integrates with client)
cif_strings = random_rows['cif']  # Iterate directly without .tolist() for memory efficiency
atoms_delayed = [delayed(cif_string_to_atoms)(cif_str) for cif_str in cif_strings]
futures = dask.persist(*atoms_delayed)
progress(futures)


##### Get ase atoms objects

In [ ]:

collected_futures = client.compute(futures)
# note to self: make sure to add None if atoms are not read correctly. otherwise we woulnd't know which atom is which.
atoms_list = [f.result() if f.status=='finished' else None for f in collected_futures]

print(f"Loaded and parsed {len(atoms_list)} random atoms from Parquet CIF strings (fast!).")

### Segmentation Wrappers
Lets wrap around segmentation functions to grab the logging output and convert them into separate times for each step of the workflow (maxima detection, pbc segemtnation, no pbc, connections, reindexing etc) and store them along with the outputs. 

In [ ]:
import logging
from io import StringIO
import time
import mofography as mgr
logging.basicConfig(level=logging.INFO)

def wrapper_segmentation_rust(index, mat_atoms, spacing):
    # Time dgrid computation
    dgrid_start = time.time()
    dgrid = mgr.dgrid_from_atoms_rust(mat_atoms, spacing=spacing)
    dgrid_time = time.time() - dgrid_start
    
    # Set up logging capture for segmentation
    log_capture_string = StringIO()
    ch = logging.StreamHandler(log_capture_string)
    ch.setLevel(logging.INFO)
    logger = logging.getLogger('mofography.rust_fun')
    logger.addHandler(ch)
    logger.setLevel(logging.INFO)
    
    # Time segmentation
    seg_start = time.time()
    result = mgr.do_periodic_segmentation_rust(dgrid, mat_atoms, spacing=spacing, extra_padding=0)
    seg_time = time.time() - seg_start
    
    # Get logs
    log_contents = log_capture_string.getvalue()
    logger.removeHandler(ch)
    
    # Add to result
    result['logs'] = log_contents
    # these are key! 
    result['material_index'] = index
    result['version'] = 'Rust'
    result['dgrid_time'] = dgrid_time
    result['seg_time'] = seg_time
    result['num_atoms'] = len(mat_atoms)
    
    return result

def wrapper_segmentation_python(index, mat_atoms, spacing):
    # Time dgrid computation
    dgrid_start = time.time()
    dgrid = mgr.dgrid_from_atoms_cpu(mat_atoms, spacing=spacing)
    dgrid_time = time.time() - dgrid_start
    
    # Set up logging capture for segmentation
    log_capture_string = StringIO()
    ch = logging.StreamHandler(log_capture_string)
    ch.setLevel(logging.INFO)
    logger = logging.getLogger('mofography.rust_fun')
    logger.addHandler(ch)
    logger.setLevel(logging.INFO)
    
    # Time segmentation
    seg_start = time.time()
    result = mgr.do_periodic_segmentation(dgrid, mat_atoms, spacing=spacing, extra_padding=0)
    seg_time = time.time() - seg_start
    
    # Get logs
    log_contents = log_capture_string.getvalue()
    logger.removeHandler(ch)
    
    # Add to result
    result['logs'] = log_contents
    # these are key! 
    result['material_index'] = index
    result['version'] = 'Python'
    result['dgrid_time'] = dgrid_time
    result['seg_time'] = seg_time
    result['num_atoms'] = len(mat_atoms)
    
    return result

### Reassign workers
Now that we are done iwth reading in the CIFs, we need to reassign the workers to have more RAM per worker to do the actual compute. Lets do the default n workers ( 4  cpus or cores on my laptop). Change as needed as per your system. when running on a cluster put all cpus in the same node, that should make data transfer easy. 

In [ ]:
client.shutdown() # shutdown old client
client = Client() # keep default on laptop, on cluster use num_cpus. 

In [ ]:
client

## Benchmarking
Well breakdown the flow into two 1. dgrid computing and 2. periodic segmentation

### RUst benchmarking

In [ ]:
delayed_results_rust= []
spacing = 0.5
for i, mat_atoms in enumerate(atoms_list):
    delayed_results_rust.append(delayed(wrapper_segmentation_rust)(i, mat_atoms, spacing))

Test one calculation

In [ ]:
print(delayed_results_rust[0].compute()['logs'])

### execute rust benchmarks
Let's run the workflow, prettyr much same steps as reading inthe cif files

In [ ]:
futures = dask.persist(*delayed_results_rust)
progress(futures)

In [ ]:
progress(futures)

wait for all to finish...

In [ ]:
collected_futures = client.compute(futures)
collected_results_rust = [f.result() if f.status=='finished' else None for f in collected_futures]

In [ ]:
print(collected_results_rust[0]['logs'])

### Plot any result
This is quick fucntion to visualize any result from the results list.

In [ ]:
def show_fig(results_rust, mat_atoms):
    import plotly.graph_objects as go
    import plotly.express as px
    import numpy as np
    mat_updated=mgr.update_with_covalent_bonds(mat_atoms)
    dgrid = results_rust['dgrid']
    maxima_info = results_rust['maxima_info']
    unique_regions = results_rust['unique_regions']
    regions = results_rust['regions']
    maxima_pbc_dict = results_rust['maxima_pbc_dict']
    maxima_pbc = results_rust['maxima_pbc']
    connections = results_rust['connections']
    all_maxima= results_rust['all_maxima']
    # all_maxima= maxima_info[:, 1:4] + maxima_info[:, 4:7] * np.array(dgrid.shape)
    pbc_groups = results_rust['pbc_groups']
    maxima=all_maxima[:regions.max()]
    regions_pbc = results_rust['regions_pbc']
    close_maxima= results_rust['close_or_inside_maxima']
    # regions_padded = results_rust['regions_padded']
    close_or_inside_regions = results_rust['close_or_inside_regions']
    # connections_trimmed = np.vstack([c for c in connections if np.any(np.isin(c[0],unique_regions))])
    connections_trimmed = [c for c in connections if np.any(np.isin(c[0],close_or_inside_regions))]
    #%%
    surface_value = 1.0
    # cscale = px.colors.sequential.Turbo
    cscale = px.colors.qualitative.Pastel
    cell_trace = mgr.plot_cell(mat_atoms)
    frame_trace = mgr.plot_framework_as_mesh(mat_updated, stylize=True)
    close_maxima_trace = mgr.plot_maxima( dgrid,close_maxima, mat_atoms, name='Close maxima', show_spheres=False, marker_colorscale=cscale)
    maxima_trace = mgr.plot_maxima( dgrid,maxima, mat_atoms, name='maxima influencing box', show_spheres=False, marker_colorscale=cscale)
    maxima_trace_pbc = mgr.plot_maxima(
        dgrid,maxima_pbc, mat_atoms, name='Maxima PBC', show_spheres=False, marker_colorscale=cscale
    )
    surface_trace_pbc=mgr.plot_surface_dgrid(
        dgrid,
        isovalue=surface_value, 
        region_labels=regions_pbc,
        ase_atoms=mat_atoms,
        cscale=cscale,
        name='Regions PBC at ' + str(surface_value) + '\u212b',
        stylize=True,
    )
    connections_trace=mgr.plot_connections_as_it_is(connections_trimmed, all_maxima, regions.shape, mat_atoms, cscale=cscale, plot_disconnected=False)
    surface_trace =  mgr.plot_surface_dgrid(
        dgrid,
        isovalue=surface_value,
        region_labels=regions,
        ase_atoms=mat_atoms,
        cscale=cscale,
        name='Regions at ' + str(surface_value) + '\u212b',
        stylize=True,
    )
    # surface_trace.lighting = mgr.default_mesh_lighting
    windows_trace = mgr.plot_windows(connections_trimmed, marker_color='blue')
    fig = go.Figure(
        # data=[cell_trace, surface_trace, *rag_trace, maxima_trace, windows_trace],
        data=[
            cell_trace,
            surface_trace_pbc,
            surface_trace,
            *connections_trace,
            close_maxima_trace,
            maxima_trace,
            maxima_trace_pbc,
            windows_trace,
            *frame_trace,
        ],
        layout=mgr.dark_layout,
    )
    # fig.update_layout(height=800, width=900)
    # fig.show(renderer='browser')
    # make_fig_orthographic(fig).show(renderer='browser')
    # fig.update_layout(height=900, width=900)
    fig.update_layout(scene_aspectmode="data")
    fig.update_layout(legend_groupclick="toggleitem")
    fig.update_layout(scene_aspectmode="data")
    # fig = mgr.make_fig_orthographic(fig)
    return fig

In [ ]:
index = 5
fig = show_fig(collected_results_rust[index], atoms_list[index])
fig.show()

### Python benchmarks
Now lets do the same for python, 1. dgrid_from_atoms_cpu 2. do_periodic_segmentation

#### workflow

In [ ]:
delayed_results_python= []
spacing = 0.5
for i, mat_atoms in enumerate(atoms_list):
    delayed_results_python.append(delayed(wrapper_segmentation_python)(i, mat_atoms, spacing))

Test one calc

In [ ]:
delayed_results_python[0].compute()

### execute python benchmarks
Let's run the workflow, prettyr much same steps as in rust flow

In [ ]:
futures = dask.persist(*delayed_results_python)
progress(futures)

In [ ]:
progress(futures)

wait for all to finish

In [ ]:
collected_futures = client.compute(futures)
collected_results_python = [f.result() if f.status=='finished' else None for f in collected_futures]

lets check one to see if the times are there

In [ ]:
print(collected_results_python[0]['version'],collected_results_python[0]['dgrid_time'], collected_results_python[0]['seg_time'])

## Plot benchmark times vs Num atoms

In [ ]:
# Create separate DataFrames
python_df = pd.DataFrame(collected_results_python)[['num_atoms', 'material_index', 'dgrid_time', 'seg_time', 'version']]
rust_df = pd.DataFrame(collected_results_rust)[['num_atoms', 'material_index', 'dgrid_time', 'seg_time', 'version']]

# Concatenate them vertically (stack them)
benchmark_df = pd.concat([python_df, rust_df], ignore_index=True)

print(f"Combined benchmarking data: {len(benchmark_df)} entries")
print(f"Number of unique materials: {benchmark_df['material_index'].nunique()}")
print(f"Expected entries: {benchmark_df['material_index'].nunique() * 2}")
print(benchmark_df.head(20))  # Show first 10 rows to verify Python/Rust pairs

In [ ]:
# Create comprehensive benchmarking plots
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Melt the data for easier plotting
melted_df = benchmark_df.melt(id_vars=['material_index', 'num_atoms', 'version'],
                              value_vars=['dgrid_time', 'seg_time'],
                              var_name='operation', value_name='time')

# Add volume column to melted_df
melted_df['volume'] = melted_df['material_index'].apply(lambda idx: atoms_list[idx].get_volume() if atoms_list[idx] is not None else None)

# 1. Scatter plot: Time vs Number of Atoms with independent scales, colored by volume
fig1 = make_subplots(rows=1, cols=2, 
                     subplot_titles=('Dgrid Time vs Number of Atoms', 'Segmentation Time vs Number of Atoms'),
                     shared_xaxes=True)

# Add traces for dgrid_time
dgrid_data = melted_df[melted_df['operation'] == 'dgrid_time']
for version in ['Python', 'Rust']:
    version_data = dgrid_data[dgrid_data['version'] == version]
    show_colorbar = (version == 'Python')  # Show colorbar only for first trace
    fig1.add_trace(go.Scatter(x=version_data['num_atoms'], y=version_data['time'], 
                             mode='markers', name=f'{version} Dgrid',
                             marker=dict(color=version_data['volume'], 
                                        colorscale='jet', 
                                        size=10,
                                        symbol='circle' if version == 'Python' else 'square',
                                        line=dict(color='black', width=0.5),
                                        colorbar=dict(title="Volume (Å³)", 
                                                     x=1.1, y=0.4, 
                                                     orientation='v',
                                                     len=0.8, thickness=20) if show_colorbar else None)),
                   row=1, col=1)

# Add traces for seg_time
seg_data = melted_df[melted_df['operation'] == 'seg_time']
for version in ['Python', 'Rust']:
    version_data = seg_data[seg_data['version'] == version]
    fig1.add_trace(go.Scatter(x=version_data['num_atoms'], y=version_data['time'], 
                             mode='markers', name=f'{version} Segmentation',
                             marker=dict(color=version_data['volume'], 
                                        colorscale='jet', 
                                        size=10,
                                        symbol='circle' if version == 'Python' else 'square',
                                        line=dict(color='black', width=0.5))),
                   row=1, col=2)

fig1.update_layout(height=500, title_text="Benchmarking: Python vs Rust Performance (Colored by Volume)",
                   template='plotly_white', showlegend=True)
fig1.update_xaxes(title_text="Number of Atoms", row=1, col=1)
fig1.update_xaxes(title_text="Number of Atoms", row=1, col=2)
fig1.update_yaxes(title_text="Time (seconds)", row=1, col=1)
fig1.update_yaxes(title_text="Time (seconds)", row=1, col=2)

# Add borders to subplots
fig1.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig1.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)

# fig1.show()

# 2. Box plot: Distribution of times by operation and version (keeping original coloring)
fig2 = px.box(melted_df, x='operation', y='time', color='version',
              title='Time Distribution by Operation and Implementation',
              labels={'operation': 'Operation', 'time': 'Time (seconds)',
                      'version': 'Implementation'},
              template='plotly_white',
              color_discrete_map={'Python': 'blue', 'Rust': 'red'})

fig2.update_layout(height=500, font=dict(size=12))
fig2.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig2.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)

fig2.show()

# 3. Performance ratio plot (Rust speedup) colored by volume
speedup_melted['volume'] = speedup_melted['material_index'].apply(lambda idx: atoms_list[idx].get_volume() if atoms_list[idx] is not None else None)

fig3 = px.scatter(speedup_melted, x='num_atoms', y='speedup', color='volume',
                  title='Rust Speedup Factor vs Number of Atoms (Colored by Volume)',
                  labels={'num_atoms': 'Number of Atoms', 'speedup': 'Speedup (Python time / Rust time)',
                          'operation': 'Operation'},
                  template='plotly_white',
                  color_continuous_scale='jet')

fig3.update_traces(marker=dict(size=10))  # Set marker size to 10

fig3.add_hline(y=1, line_dash="dash", line_color="red", annotation_text="No speedup")
fig3.update_layout(height=500, font=dict(size=12))
fig3.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig3.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)

# Update colorbar position
fig3.update_coloraxes(colorbar=dict(title="Volume (Å³)", orientation='v', y=0.4, x=1.1, len=0.8))

fig3.show()

# 4. Combined subplot view with professional styling, colored by volume
fig4 = make_subplots(rows=2, cols=2,
                     subplot_titles=('Dgrid Time vs Atoms', 'Segmentation Time vs Atoms',
                                   'Dgrid Speedup', 'Segmentation Speedup'),
                     specs=[[{"secondary_y": False}, {"secondary_y": False}],
                            [{"secondary_y": False}, {"secondary_y": False}]])

# Add volume to benchmark_df for coloring
benchmark_df['volume'] = benchmark_df['material_index'].apply(lambda idx: atoms_list[idx].get_volume() if atoms_list[idx] is not None else None)

# Add volume to speedup_df for coloring
speedup_df['volume'] = speedup_df['material_index'].apply(lambda idx: atoms_list[idx].get_volume() if atoms_list[idx] is not None else None)

# Add traces for time comparisons
for version in ['Python', 'Rust']:
    data = benchmark_df[benchmark_df['version'] == version]
    symbol = 'circle' if version == 'Python' else 'square'
    show_colorbar = (version == 'Python')  # Show colorbar only for first trace
    fig4.add_trace(go.Scatter(x=data['num_atoms'], y=data['dgrid_time'], mode='markers',
                             name=f'{version} Dgrid', legendgroup=version,
                             marker=dict(color=data['volume'], colorscale='jet', 
                                        size=10, symbol=symbol,
                                        line=dict(color='black', width=0.5),
                                        colorbar=dict(title="Volume (Å³)", 
                                                     x=1.1, y=0.4, 
                                                     orientation='v',
                                                     len=0.8, thickness=20) if show_colorbar else None)), row=1, col=1)
    fig4.add_trace(go.Scatter(x=data['num_atoms'], y=data['seg_time'], mode='markers',
                             name=f'{version} Seg', legendgroup=version,
                             marker=dict(color=data['volume'], colorscale='jet', 
                                        size=10, symbol=symbol,
                                        line=dict(color='black', width=0.5))), row=1, col=2)

# Add traces for speedup
fig4.add_trace(go.Scatter(x=speedup_df['num_atoms'], y=speedup_df['dgrid_speedup'],
                         mode='markers', name='Dgrid Speedup',
                         marker=dict(color=speedup_df['volume'], colorscale='jet', 
                                    size=10, line=dict(color='black', width=0.5))), row=2, col=1)
fig4.add_trace(go.Scatter(x=speedup_df['num_atoms'], y=speedup_df['seg_speedup'],
                         mode='markers', name='Seg Speedup',
                         marker=dict(color=speedup_df['volume'], colorscale='jet', 
                                    size=10, line=dict(color='black', width=0.5))), row=2, col=2)

fig4.update_layout(height=800, title_text="Comprehensive Benchmarking Dashboard (Colored by Volume)",
                   template='plotly_white', font=dict(size=12))

# Update axes with borders
for row in [1, 2]:
    for col in [1, 2]:
        fig4.update_xaxes(title_text="Number of Atoms", row=row, col=col,
                         showline=True, linewidth=2, linecolor='black', mirror=True)
        if row == 1:
            fig4.update_yaxes(title_text="Time (s)", row=row, col=col,
                             showline=True, linewidth=2, linecolor='black', mirror=True)
        else:
            fig4.update_yaxes(title_text="Speedup Factor", row=row, col=col,
                             showline=True, linewidth=2, linecolor='black', mirror=True)

fig4.show()